<a href="https://colab.research.google.com/github/tettyvera/loan-portfolio-risk-analytics/blob/main/Outstanding_and_Delinquency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Outstanding Principal & Delinquency

**Project:** Loan Portfolio Performance & Risk Analytics  
**SQL:** DuckDB running inside Google Colab  
**Snapshot:** 31 August 2026 (end of day)  
**Currency:** IDR (Indonesian rupiah)  
**Data:** entirely synthetic


## Step 1 — Start DuckDB

In [1]:
%pip -q install duckdb
import duckdb
con = duckdb.connect(database=":memory:")
print("DuckDB ready:", duckdb.__version__)

DuckDB ready: 1.3.2


## Step 2 — Upload the five CSV files

In [2]:
from google.colab import files
uploaded = files.upload()
required = {"borrowers.csv", "lenders.csv", "loans.csv", "installments.csv", "repayments.csv"}
missing = required - set(uploaded)
if missing:
    raise ValueError(f"Please upload the missing file(s): {sorted(missing)}")
for name in ["borrowers", "lenders", "loans", "installments", "repayments"]:
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv_auto('{name}.csv')")
print("Five tables are ready!")


Saving borrowers.csv to borrowers.csv
Saving installments.csv to installments.csv
Saving lenders.csv to lenders.csv
Saving loans.csv to loans.csv
Saving repayments.csv to repayments.csv
Five tables are ready!


## Step 3 — Build the loan snapshot (three SQL views)

### 1. Aggregate repayments per installment

In [3]:
con.execute("""
-- STEP 1. Aggregate repayments FIRST: otherwise joining each loan to many
-- installments / payments can duplicate the loan principal.
CREATE OR REPLACE VIEW v_installment_payments AS
SELECT
    installment_id,
    SUM(CAST(principal_paid AS DECIMAL(18, 2))) AS principal_paid,
    SUM(CAST(interest_paid AS DECIMAL(18, 2))) AS interest_paid
FROM repayments
WHERE CAST(payment_date AS DATE) <= DATE '2026-08-31'
GROUP BY installment_id;
""")
print("Created view v_installment_payments")

Created view v_installment_payments


### 2. Calculate installment-level balances & DPD

In [4]:
con.execute("""
-- STEP 2. Compare scheduled installment amounts with payments.
-- Future scheduled installments remain outstanding, but NOT overdue.
CREATE OR REPLACE VIEW v_installment_snapshot AS
WITH balances AS (
    SELECT
        i.installment_id,
        i.loan_id,
        CAST(i.due_date AS DATE) AS due_date,
        CAST(i.principal_due AS DECIMAL(18, 2)) AS principal_due,
        CAST(i.interest_due AS DECIMAL(18, 2)) AS interest_due,
        COALESCE(p.principal_paid, 0) AS principal_paid,
        COALESCE(p.interest_paid, 0) AS interest_paid,
        GREATEST(
            CAST(i.principal_due AS DECIMAL(18, 2)) - COALESCE(p.principal_paid, 0), 0
        ) AS remaining_principal,
        GREATEST(
            CAST(i.interest_due AS DECIMAL(18, 2)) - COALESCE(p.interest_paid, 0), 0
        ) AS remaining_interest
    FROM installments AS i
    LEFT JOIN v_installment_payments AS p
        ON i.installment_id = p.installment_id
)
SELECT
    *,
    CASE
        WHEN due_date < DATE '2026-08-31'
             AND remaining_principal + remaining_interest > 0
        THEN DATE_DIFF('day', due_date, DATE '2026-08-31')
        ELSE 0
    END AS installment_dpd,
    CASE
        WHEN due_date < DATE '2026-08-31'
        THEN remaining_principal
        ELSE 0
    END AS overdue_principal
FROM balances;
""")
print("Created view v_installment_snapshot")

Created view v_installment_snapshot


### 3. Aggregate into one loan snapshot

In [5]:
con.execute("""
-- STEP 3. Make ONE row per loan (safe to join to loan dimensions).
CREATE OR REPLACE VIEW v_loan_snapshot AS
WITH rollup AS (
    SELECT
        loan_id,
        SUM(remaining_principal) AS outstanding_principal,
        SUM(overdue_principal) AS overdue_principal,
        MAX(installment_dpd) AS dpd
    FROM v_installment_snapshot
    GROUP BY loan_id
)
SELECT
    l.loan_id,
    l.borrower_id,
    l.lender_id,
    CAST(l.disbursement_date AS DATE) AS disbursement_date,
    CAST(l.principal_amount AS DECIMAL(18, 2)) AS disbursed_principal,
    COALESCE(r.outstanding_principal, 0) AS outstanding_principal,
    COALESCE(r.overdue_principal, 0) AS overdue_principal,
    COALESCE(r.dpd, 0) AS dpd,
    CASE
        WHEN COALESCE(r.outstanding_principal, 0) = 0 THEN 'Paid off'
        WHEN COALESCE(r.dpd, 0) = 0 THEN 'Current'
        WHEN r.dpd BETWEEN 1 AND 30 THEN '1-30'
        WHEN r.dpd BETWEEN 31 AND 60 THEN '31-60'
        WHEN r.dpd BETWEEN 61 AND 89 THEN '61-89'
        ELSE '90+'
    END AS dpd_bucket
FROM loans AS l
LEFT JOIN rollup AS r ON l.loan_id = r.loan_id
WHERE CAST(l.disbursement_date AS DATE) <= DATE '2026-08-31';
""")
print("Created view v_loan_snapshot")

Created view v_loan_snapshot


### Quick preview — one row per loan

In [6]:
con.sql("SELECT * FROM v_loan_snapshot ORDER BY loan_id LIMIT 10").df()

,loan_id,borrower_id,lender_id,disbursement_date,disbursed_principal,outstanding_principal,overdue_principal,dpd,dpd_bucket
0,LN000001,B01102,L002,2026-07-07,28300000.0,25941666.66,0.00,0,Current
1,LN000002,B00122,L003,2024-05-28,16900000.0,5633333.33,5633333.33,733,90+
2,LN000003,B00086,L003,2025-09-28,11200000.0,1866666.67,933333.34,246,90+
3,LN000004,B01340,L001,2025-10-22,21600000.0,15600000.00,6000000.00,282,90+
4,LN000005,B00721,L003,2024-11-08,5400000.0,540000.00,540000.00,449,90+
5,LN000006,B00920,L002,2025-05-30,13000000.0,4333333.33,4333333.33,427,90+
6,LN000007,B00623,L003,2025-09-07,15500000.0,7750000.00,7750000.00,297,90+
7,LN000008,B01168,L004,2025-09-17,6900000.0,920000.00,345000.00,167,90+
8,LN000009,B01221,L002,2024-05-30,16800000.0,7466666.67,7466666.67,762,90+
9,LN000010,B00032,L004,2026-01-28,22300000.0,4955555.54,0.00,0,Current


## Step 4 — Portfolio health: disbursement, outstanding & PAR

In [7]:
con.sql("""
-- QUESTION A: Overall portfolio health
-- PAR >30 = SUM(outstanding on loans with DPD > 30) / total outstanding
-- PAR 90+ = SUM(outstanding on loans with DPD >= 90) / total outstanding
SELECT
    COUNT(*) AS total_loans,
    COUNT(*) FILTER (WHERE outstanding_principal > 0) AS loans_with_outstanding,
    SUM(disbursed_principal) AS cumulative_disbursement_idr,
    SUM(outstanding_principal) AS outstanding_principal_idr,
    SUM(overdue_principal) AS overdue_principal_idr,
    COUNT(*) FILTER (WHERE dpd > 0) AS delinquent_loans,
    COUNT(*) FILTER (WHERE dpd > 30) AS loans_dpd_over_30,
    ROUND(
        100.0 * SUM(CASE WHEN dpd > 30 THEN outstanding_principal ELSE 0 END)
        / NULLIF(SUM(outstanding_principal), 0), 2
    ) AS par_over_30_pct,
    ROUND(
        100.0 * SUM(CASE WHEN dpd >= 90 THEN outstanding_principal ELSE 0 END)
        / NULLIF(SUM(outstanding_principal), 0), 2
    ) AS par_90_plus_pct
FROM v_loan_snapshot;
""").df()

,total_loans,loans_with_outstanding,cumulative_disbursement_idr,outstanding_principal_idr,overdue_principal_idr,delinquent_loans,loans_dpd_over_30,par_over_30_pct,par_90_plus_pct
0,2000,1648,3.170440e+10,9.592801e+09,4.174171e+09,1358,1286,59.98,48.89


### DPD bucket distribution

In [8]:
con.sql("""
-- QUESTION B: What is the outstanding distribution across DPD buckets?
SELECT
    dpd_bucket,
    COUNT(*) AS loan_count,
    SUM(outstanding_principal) AS outstanding_principal_idr,
    SUM(overdue_principal) AS overdue_principal_idr,
    ROUND(
        100.0 * SUM(outstanding_principal)
        / NULLIF((SELECT SUM(outstanding_principal) FROM v_loan_snapshot), 0), 2
    ) AS share_of_total_outstanding_pct
FROM v_loan_snapshot
GROUP BY dpd_bucket
ORDER BY CASE dpd_bucket
    WHEN 'Paid off' THEN 0 WHEN 'Current' THEN 1
    WHEN '1-30' THEN 2 WHEN '31-60' THEN 3
    WHEN '61-89' THEN 4 ELSE 5 END;
""").df()

,dpd_bucket,loan_count,outstanding_principal_idr,overdue_principal_idr,share_of_total_outstanding_pct
0,Paid off,352,0.000000e+00,0.000000e+00,0.00
1,Current,290,3.119631e+09,0.000000e+00,32.52
2,1-30,72,7.189685e+08,1.119268e+08,7.49
3,31-60,49,5.201254e+08,9.938653e+07,5.42
4,61-89,57,5.443268e+08,1.253379e+08,5.67
5,90+,1180,4.689750e+09,3.837519e+09,48.89


### Lender risk comparison


In [9]:
con.sql("""
-- QUESTION C: Which lender has a higher share of delinquent outstanding?
SELECT
    ld.lender_name,
    COUNT(*) AS total_loans,
    SUM(s.disbursed_principal) AS cumulative_disbursement_idr,
    SUM(s.outstanding_principal) AS outstanding_principal_idr,
    ROUND(
        100.0 * SUM(CASE WHEN s.dpd > 30 THEN s.outstanding_principal ELSE 0 END)
        / NULLIF(SUM(s.outstanding_principal), 0), 2
    ) AS par_over_30_pct
FROM v_loan_snapshot AS s
JOIN lenders AS ld ON s.lender_id = ld.lender_id
GROUP BY ld.lender_name
ORDER BY par_over_30_pct DESC;
""").df()

,lender_name,total_loans,cumulative_disbursement_idr,outstanding_principal_idr,par_over_30_pct
0,Harbor Finance,417,6.575000e+09,2.212104e+09,68.50
1,Summit Credit,261,4.247600e+09,1.367469e+09,64.17
2,Crescent Lending,533,8.381800e+09,2.627206e+09,57.74
3,Meridian Capital,789,1.250000e+10,3.386021e+09,54.47


### Borrower risk segment

In [10]:
con.sql("""
-- QUESTION D: Which predefined risk segments have the highest PAR >30?
SELECT
    b.risk_segment,
    COUNT(*) AS total_loans,
    SUM(s.outstanding_principal) AS outstanding_principal_idr,
    ROUND(
        100.0 * SUM(CASE WHEN s.dpd > 30 THEN s.outstanding_principal ELSE 0 END)
        / NULLIF(SUM(s.outstanding_principal), 0), 2
    ) AS par_over_30_pct
FROM v_loan_snapshot AS s
JOIN borrowers AS b ON s.borrower_id = b.borrower_id
GROUP BY b.risk_segment
ORDER BY par_over_30_pct DESC;
""").df()

,risk_segment,total_loans,outstanding_principal_idr,par_over_30_pct
0,High,222,1.749031e+09,85.25
1,Medium,720,3.726970e+09,66.29
2,Low,1058,4.116800e+09,43.54


### Check impossible installment balances

In [11]:
con.sql("""
-- QUALITY CHECK A: no overpaid installments / negative unpaid balances.
SELECT
    COUNT(*) FILTER (WHERE remaining_principal < 0 OR remaining_interest < 0)
        AS negative_installment_balances,
    COUNT(*) FILTER (WHERE principal_paid > principal_due OR interest_paid > interest_due)
        AS overpaid_installments
FROM v_installment_snapshot;
""").df()

,negative_installment_balances,overpaid_installments
0,0,0


### Reconcile portfolio principal

`reconciliation_difference_idr` should equal exactly **0.00**.

In [12]:
con.sql("""
-- QUALITY CHECK B: total disbursement less total principal paid
-- must equal outstanding from the installment schedule (to within IDR 0.01).
SELECT
    SUM(s.disbursed_principal) AS disbursed_idr,
    (SELECT SUM(CAST(principal_paid AS DECIMAL(18,2)))
       FROM repayments WHERE CAST(payment_date AS DATE) <= DATE '2026-08-31')
        AS principal_paid_idr,
    SUM(s.outstanding_principal) AS outstanding_idr,
    SUM(s.disbursed_principal) -
    (SELECT SUM(CAST(principal_paid AS DECIMAL(18,2)))
       FROM repayments WHERE CAST(payment_date AS DATE) <= DATE '2026-08-31') -
    SUM(s.outstanding_principal) AS reconciliation_difference_idr
FROM v_loan_snapshot AS s;
""").df()

,disbursed_idr,principal_paid_idr,outstanding_idr,reconciliation_difference_idr
0,3.170440e+10,2.211160e+10,9.592801e+09,0.0


## Reference results for the provided synthetic dataset

- Cumulative disbursement: approximately **IDR 31.704 billion**.
- Outstanding principal: approximately **IDR 9.593 billion**.
- Outstanding on loans with DPD >30: approximately **59.98%** of total outstanding.
- Outstanding on loans with DPD ≥90: approximately **48.89%** of total outstanding.
- Highest cumulative disbursement lender: **Meridian Capital**.
- Highest PAR >30 lender: **Harbor Finance**.

These are **simulation results**, not evidence about any real lender or lending market. The generator intentionally produces severe delinquency scenarios to make the analytical exercise meaningful.
